# HW6: 총액 회귀 모델 비교와 API 배포

박지호 · 21010856

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor


## ① 데이터 읽기와 분할

In [2]:
sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)
print("training :", len(X_train), " test :", len(X_test))


training : 803  test : 201


## ② 세 모델의 test R²

In [3]:
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])
lin = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("linear regression test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit) test R² :", round(tree.score(X_test, y_test), 3))
print("linear + qp coefficients :", lin2.coef_.round(3), "intercept :", round(lin2.intercept_, 1))


linear regression test R² : 0.862
linear regression + qp test R² : 1.0
decision tree (no limit) test R² : 0.994
linear + qp coefficients : [0. 0. 1.] intercept : -0.0


## ③ 기본 선형 회귀와 `qp`의 차이

기본 선형 회귀는 수량과 단가의 가중합만 표현하므로 곱셈으로 정해지는 총액을 정확하게 계산할 수 없다. 수량 × 단가인 `qp`를 추가하면 실제 총액 공식을 모델이 표현할 수 있다.

## ④ 트리 깊이별 잎 수와 R²

In [4]:
rows = []
for depth in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_train, y_train)
    rows.append([str(depth), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3),
                 round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])


,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## ⑤ 5-fold 교차검증과 깊이 선택

In [5]:
for depth in [3, 5, 8, 12, None]:
    scores = cross_val_score(
        DecisionTreeRegressor(max_depth=depth, random_state=0), X, y, cv=5
    )
    print(f"max_depth {str(depth):4s} mean {scores.mean():.3f} std {scores.std():.3f}")


max_depth 3    mean 0.845 std 0.019
max_depth 5    mean 0.961 std 0.003
max_depth 8    mean 0.992 std 0.000
max_depth 12   mean 0.995 std 0.000
max_depth None mean 0.994 std 0.001


트리 모델을 쓴다면 `max_depth=8`을 고른다. depth 12보다 교차검증 평균 R²가 0.003 낮지만, 잎이 250개로 781개보다 훨씬 적다. 실제 총액 규칙이 일정하므로 제한 없는 트리도 training R²와 test R²가 가깝다.

## ⑥ API용 모델 저장

최종 API에는 `qp`를 더한 선형 회귀를 사용한다. 비교가 끝난 뒤 전체 1,004행으로 다시 학습해 저장한다.

In [6]:
X2 = X.assign(qp=X["quantity"] * X["unit_price"])
sales_model = LinearRegression().fit(X2, y)
joblib.dump(sales_model, "sales_model.pkl")
sample = pd.DataFrame([[40, 200000]], columns=["quantity", "unit_price"])
sample["qp"] = sample["quantity"] * sample["unit_price"]
print("linear + qp prediction :", int(round(sales_model.predict(sample)[0])))


linear + qp prediction : 8000000


## ⑦ 모델 선택 근거

In [7]:
tree8 = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)
print("linear + qp :", int(round(sales_model.predict(sample)[0])))
print("tree depth 8 :", int(round(tree8.predict(sample[["quantity", "unit_price"]])[0])))
print("quantity × unit_price :", 40 * 200000)


linear + qp : 8000000
tree depth 8 : 7362301
quantity × unit_price : 8000000


`qp`를 추가한 선형 회귀는 실제 곱셈 결과인 8,000,000원을 정확히 예측한다. depth 8 트리는 학습 데이터의 구간별 값을 근사하므로 이 입력에서 차이가 난다. 따라서 곱셈 규칙이 명확한 이 과제에는 `qp` 선형 회귀가 적합하다.